# Lab 04 — LLM Calls: Conversations and System Prompts

**Knowledge base:** `knowledge_base/02_llms/01_how_llms_work.md`

**Concepts:** `generate_with_multiple_input` · message history · system prompts · chat function

This lab is entirely focused on `generate_with_multiple_input`.
Understanding this function is the prerequisite for building the full RAG loop in lab05.

---

In [ ]:
import os, sys
sys.path.insert(0, ".")
from dotenv import load_dotenv
from utils import generate_with_multiple_input

load_dotenv(override=True)
print(f"✅ Backend: {os.environ.get('LLM_BACKEND', 'ollama')}")

---
## 1 — The messages list

`generate_with_multiple_input` takes a list of message dicts.
The model sees the **entire list** and uses all prior turns as context.

Each message has exactly two keys:
- `role`: `'system'` · `'user'` · `'assistant'`
- `content`: the text of the message

In [ ]:
# Two turns — the model knows the context from turn 1 when it answers turn 2
messages = [
    {"role": "user",      "content": "The Nile River is approximately 6,650 km long."},
    {"role": "assistant", "content": "That makes it one of the longest rivers in the world."},
    {"role": "user",      "content": "How does its length compare to the Amazon?"},
]

output = generate_with_multiple_input(messages=messages, max_tokens=100)
print(output['content'])

---
## 2 — System messages

A system message sets the model's **persona, constraints, and tone**
for the entire conversation. It is always the first message in the list.

In [ ]:
# System message: force one-sentence answers
messages = [
    {"role": "system", "content": "You are a concise assistant. Answer in one sentence only."},
    {"role": "user",   "content": "What is Retrieval Augmented Generation?"}
]

output = generate_with_multiple_input(messages=messages)
print(output['content'])

In [ ]:
# System message for a RAG assistant — the pattern you will use in lab05
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful assistant. "
            "Answer the user's question using ONLY the provided context. "
            "If the context does not contain the answer, say: 'I don't know based on the provided documents.'"
        )
    },
    {"role": "user", "content": "Context: The Suez Canal connects the Red Sea to the Mediterranean Sea.\n\nQuestion: What does the Suez Canal connect?"}
]

output = generate_with_multiple_input(messages=messages, max_tokens=80)
print(output['content'])

---
## 3 — Building conversation history dynamically

In a real RAG chatbot, you don't hardcode the messages list.
You start with a system message and grow the list as the conversation continues.

In [ ]:
def chat(user_message: str, history: list, max_tokens: int = 150) -> str:
    """
    Append a user message to history, call the LLM, append the response, return the answer.

    This function modifies history in-place — the list grows with each turn.

    Args:
        user_message: The user's latest message.
        history:      The running list of message dicts (modified in place).
        max_tokens:   Max response length.

    Returns:
        The assistant's answer as a plain string.
    """
    history.append({"role": "user", "content": user_message})
    response = generate_with_multiple_input(messages=history, max_tokens=max_tokens)
    history.append({"role": "assistant", "content": response["content"]})
    return response["content"]


# Initialize history with a system message
history = [
    {"role": "system", "content": "You are a knowledgeable assistant teaching RAG concepts."}
]

# Have a 3-turn conversation
answer1 = chat("What are the three core components of a RAG system?", history)
print("Turn 1:", answer1)
print()

answer2 = chat("Which of those three is the most important for result quality?", history)
print("Turn 2:", answer2)
print()

answer3 = chat("How many turns have we had so far in this conversation?", history)
print("Turn 3:", answer3)

In [ ]:
# Inspect the full history — see how it grew
print(f"History has {len(history)} messages:\n")
for i, msg in enumerate(history):
    role = msg['role'].upper()
    preview = msg['content'][:80].replace('\n', ' ')
    print(f"  [{i}] {role:10s}: {preview}...")

---
## 4 — Why message history matters for RAG

In a RAG chatbot, the conversation history lets the user ask follow-up questions
without repeating themselves. The model remembers what was retrieved and answered.

In [ ]:
# Simulated RAG conversation — hardcoded retrieval, real LLM calls
RETRIEVED_CONTEXT = """
[Doc 1] The Suez Canal, opened in 1869, is 193 km long and connects the Red Sea to the Mediterranean.
[Doc 2] It was built under the direction of Ferdinand de Lesseps and took 10 years to construct.
[Doc 3] The canal was nationalized by Egypt in 1956, triggering the Suez Crisis.
"""

rag_history = [
    {
        "role": "system",
        "content": (
            "You are a RAG assistant. Answer using ONLY the documents provided in the conversation. "
            "If the documents don't contain the answer, say so."
        )
    },
    {
        "role": "user",
        "content": f"Context documents:\n{RETRIEVED_CONTEXT}\n\nQuestion: When was the Suez Canal opened?"
    }
]

# First turn — uses retrieved context
a1 = chat("", rag_history)   # initial context already added above
# Actually let's do it properly:
rag_history2 = [
    {
        "role": "system",
        "content": (
            "You are a RAG assistant. Answer using ONLY the documents below.\n\n"
            f"Available documents:\n{RETRIEVED_CONTEXT}"
        )
    }
]

q1 = chat("When was the Suez Canal opened?", rag_history2, max_tokens=60)
print("Q1:", q1)

q2 = chat("Who built it?", rag_history2, max_tokens=60)
print("\nQ2 (follow-up — model uses history to know what 'it' refers to):", q2)

q3 = chat("What happened to it in 1956?", rag_history2, max_tokens=80)
print("\nQ3:", q3)

---
## 5 — Exercise

Build a simple customer support chatbot using `chat()` and a system prompt.
The chatbot should:
1. Only answer questions about a product described in the system message
2. Politely refuse questions outside that topic

In [ ]:
# Fill in the system message and test the chatbot with at least 2 questions

PRODUCT_INFO = """
Product: ProScan X1 Barcode Scanner
- Reads 1D and 2D barcodes
- USB and Bluetooth connectivity
- 6-hour battery life
- Compatible with Windows, macOS, Linux
- Price: $89
"""

support_history = [
    {
        "role": "system",
        "content": f"""You are a product support assistant for the ProScan X1 barcode scanner.
Answer questions ONLY about this product using the information below.
If the question is not about this product, say: "I can only help with ProScan X1 questions."

Product information:
{PRODUCT_INFO}"""
    }
]

# YOUR CODE HERE — call chat() with at least 2 questions
# Question 1: ask something the product info answers
# Question 2: ask something completely unrelated (e.g. about weather)

print("✅ Chatbot exercise complete")

---
**Lab 04 complete.**

You can now build multi-turn conversations, write system prompts, and maintain
conversation history with the `chat()` function.

**Next:** `lab05_augmented_prompts.ipynb` — the full RAG loop: retrieve → augment → generate.